# Two-body eigenvector variety — numerical dimension

We compute the dimension of the *eigenvector variety* for the fermionic two-body operators $W$. They form a linear space $\mathcal{W}_{k,m}$ of $\binom{m}{k} \times \binom{m}{k}$ matrices which is parameterized by complex $\binom{m}{2} \times \binom{m}{2}$ symmetric matrices $w$. So in this case $n = \binom{m}{k}$ and $d = \binom{\binom{m}{2} + 1}{2}$.

For generic $w$ the eigenpair $(\lambda, x)$ of $W$ is simple. By Proposition 7.3 in our paper the eigenvector variety $\mathcal{E}(\mathcal{W}_{k,m})$ is irreducible and in fact its characteristic polynomial is also irreducible and thus squarefree. Hence by Proposition 4.2 the dimension of the eigenvector vairety equals
$$\dim \mathcal{E}(\mathcal{W}_{k,m}) = \operatorname{rank}\bigl(M(x)\bigr) - 1$$
where $M(x)$ is the matrix (3) from our paper. This is also the Jacobian of the eigenpair equations with respect to the parameteric variables $w$.

**Pipeline:**
1. Build $W$ as a sparse $\binom{m}{k}\times\binom{m}{k}$ matrix.
2. Get the leading eigenpair $(\lambda, x)$ via Arpack.
3. Build matrix $M(x)$ (size $\binom{m}{k}\times\binom{\binom{m}{2}+1}{2}$).
4. Take rank via sparse QR.

In [6]:
using Combinatorics
using LinearAlgebra
using SparseArrays
using Arpack
using Base.Threads

println("Julia threads: ", nthreads())

# ─── problem size ──────────────────────────────────────────
m = 16    # orbital number
k = 7     # particle number (we work in ∧^k ℂ^m)

n = binomial(m, k)                       # dim ∧^k ℂ^m   (rows of W, M)
d = binomial(binomial(m, 2) + 1, 2)      # dim W_{k,m} (cols of M)

println("(m, k) = ($m, $k) → n = $n,  d = $d")

Julia threads: 16
(m, k) = (16, 7) → n = 11440,  d = 7260


## Indexing tables

* `pair_idx[r,s]` (with $r<s$) indexes 2-element subsets of $[m]$.
* `dset_masks[i]` is the bitmask of the $i$-th $k$-subset.
* `mask_to_idx[mask]` is the inverse map (bitmask → row index).  Sized $2^{m}$, fine for $m\lesssim 25$.
* `wpair_col[α,β]` is the column index in $M$ for the symmetric pair $(α,β)$ of 2-element subsets.

In [7]:
pair_idx = zeros(Int, m, m)
let idx = 0
    for r in 1:m, s in r+1:m
        idx += 1
        pair_idx[r, s] = idx
    end
end

dset_masks  = Vector{UInt32}(undef, n)
mask_to_idx = zeros(Int, 1 << m)
let idx = 0
    for c in combinations(1:m, k)
        idx += 1
        mask = UInt32(0)
        for e in c
            mask |= UInt32(1) << (e - 1)
        end
        dset_masks[idx]    = mask
        mask_to_idx[mask]  = idx
    end
end
full_mask = UInt32((1 << m) - 1)

wpair_col = zeros(Int, binomial(m, 2), binomial(m, 2))
let idx = 0
    for i in 1:binomial(m, 2), j in i:binomial(m, 2)
        idx += 1
        wpair_col[i, j] = idx
        wpair_col[j, i] = idx
    end
end;

## Builders for $W(w)$ and $M(x)$

Both matrices share the same combinatorial structure: for each $k$-subset $I$ (column), enumerate pairs $(p,q)\subset I$ to remove and pairs $(r,s)\subset [m]\setminus(I\setminus\{p,q\})$ to insert, giving a new $k$-subset $J$.  The only difference is what we store at $(J, \cdot)$:

* `build_W_sparse`: column = $I$, value = $\mathrm{sgn}\cdot w_{\beta\alpha}$.
* `build_M_sparse`: column = `wpair_col[α,β]`, value = $\mathrm{sgn}\cdot x_I$.

The sign comes from re-sorting the wedge $e_I$ after the swap $(p,q)\leftrightarrow(r,s)$: the sign for pulling $e_p\wedge e_q$ to the front, times the sign for inserting $e_r\wedge e_s$ into the remaining sorted indices.

In [8]:
function build_W_sparse(w, m, k, n,
                        dset_masks, mask_to_idx, pair_idx, full_mask)
    nt        = Threads.maxthreadid()
    nnz_est   = n * binomial(k, 2) * binomial(m - k + 2, 2)
    per_thr   = nnz_est ÷ max(nt, 1) + 1
    L_rows = [sizehint!(Int[],        per_thr) for _ in 1:nt]
    L_cols = [sizehint!(Int[],        per_thr) for _ in 1:nt]
    L_vals = [sizehint!(ComplexF64[], per_thr) for _ in 1:nt]
    L_I    = [Vector{Int}(undef, k)            for _ in 1:nt]
    L_av   = [Vector{Int}(undef, m - k + 2)    for _ in 1:nt]

    @threads for col in 1:n
        tid = threadid()
        rows  = L_rows[tid]; cols = L_cols[tid]; vals = L_vals[tid]
        I_buf = L_I[tid];    a_buf = L_av[tid]
        Imask = dset_masks[col]

        # extract sorted elements of I from its bitmask
        tmp = Imask; ei = 0
        while tmp != UInt32(0)
            ei += 1
            bit = tmp & (-tmp)
            I_buf[ei] = trailing_zeros(bit) + 1
            tmp ⊻= bit
        end

        # pair (p,q) ⊂ I to remove
        for ai in 1:k-1, bi in ai+1:k
            p, q  = I_buf[ai], I_buf[bi]
            alpha = pair_idx[p, q]
            rest  = Imask ⊻ ((UInt32(1) << (p-1)) | (UInt32(1) << (q-1)))

            # available positions for new pair (r,s)
            avail = full_mask & ~rest
            tmp = avail; na = 0
            while tmp != UInt32(0)
                na += 1
                bit = tmp & (-tmp)
                a_buf[na] = trailing_zeros(bit) + 1
                tmp ⊻= bit
            end

            for ri in 1:na-1, si in ri+1:na
                r, s = a_buf[ri], a_buf[si]
                beta = pair_idx[r, s]
                J    = rest | (UInt32(1) << (r-1)) | (UInt32(1) << (s-1))
                inv  = count_ones(rest >> r) + count_ones(rest >> s)
                sgn  = isodd(ai + bi + inv) ? 1 : -1
                push!(rows, mask_to_idx[J])
                push!(cols, col)
                push!(vals, sgn * w[beta, alpha])
            end
        end
    end
    sparse(vcat(L_rows...), vcat(L_cols...), vcat(L_vals...), n, n)
end


function build_M_sparse(x, m, k, n, d,
                        dset_masks, mask_to_idx, pair_idx, full_mask, wpair_col)
    nt        = Threads.maxthreadid()
    nnz_est   = n * binomial(k, 2) * binomial(m - k + 2, 2)
    per_thr   = nnz_est ÷ max(nt, 1) + 1
    T = eltype(x)
    L_rows = [sizehint!(Int[],   per_thr) for _ in 1:nt]
    L_cols = [sizehint!(Int[],   per_thr) for _ in 1:nt]
    L_vals = [sizehint!(T[],     per_thr) for _ in 1:nt]
    L_I    = [Vector{Int}(undef, k)         for _ in 1:nt]
    L_av   = [Vector{Int}(undef, m - k + 2) for _ in 1:nt]

    @threads for col in 1:n
        tid   = threadid()
        rows  = L_rows[tid]; cols = L_cols[tid]; vals = L_vals[tid]
        I_buf = L_I[tid];    a_buf = L_av[tid]
        Imask = dset_masks[col]
        xc    = x[col]

        tmp = Imask; ei = 0
        while tmp != UInt32(0)
            ei += 1
            bit = tmp & (-tmp)
            I_buf[ei] = trailing_zeros(bit) + 1
            tmp ⊻= bit
        end

        for ai in 1:k-1, bi in ai+1:k
            p, q  = I_buf[ai], I_buf[bi]
            alpha = pair_idx[p, q]
            rest  = Imask ⊻ ((UInt32(1) << (p-1)) | (UInt32(1) << (q-1)))

            avail = full_mask & ~rest
            tmp = avail; na = 0
            while tmp != UInt32(0)
                na += 1
                bit = tmp & (-tmp)
                a_buf[na] = trailing_zeros(bit) + 1
                tmp ⊻= bit
            end

            for ri in 1:na-1, si in ri+1:na
                r, s = a_buf[ri], a_buf[si]
                beta = pair_idx[r, s]
                J    = rest | (UInt32(1) << (r-1)) | (UInt32(1) << (s-1))
                inv  = count_ones(rest >> r) + count_ones(rest >> s)
                sgn  = isodd(ai + bi + inv) ? 1 : -1
                push!(rows, mask_to_idx[J])
                push!(cols, wpair_col[alpha, beta])
                push!(vals, sgn * xc)
            end
        end
    end
    sparse(vcat(L_rows...), vcat(L_cols...), vcat(L_vals...), n, d)
end


function sparse_qr_rank(A; rtol = 1e-9)
    F  = qr(A)
    dR = abs.(diag(F.R))
    isempty(dR) && return 0
    tol = rtol * first(dR)
    return count(>(tol), dR)
end;

## Numerical pipeline

Sample a generic complex symmetric $w$, find a leading eigenpair $(\lambda, x)$ of $W(w)$, build $M(x)$, and read off the rank.

In [9]:
# 1. Random complex symmetric 2-body matrix w
R = randn(ComplexF64, binomial(m, 2), binomial(m, 2))
w = (R + transpose(R)) / 2

# 2. Assemble W(w) and grab the leading eigenpair
@time "build W" W = build_W_sparse(w, m, k, n,
                                     dset_masks, mask_to_idx, pair_idx, full_mask)
println("W: $(n)×$(n)")

vals, vecs, _, niter = eigs(W; nev = 1, ncv = min(50, n), maxiter = 1000)
λ = vals[1]
x = vecs[:, 1]
println("λ = $λ   (converged in $niter Arnoldi iters)")

# 3. Assemble M(x) — matrix (3) in the paper
@time "build M" M = build_M_sparse(x, m, k, n, d,
                                     dset_masks, mask_to_idx, pair_idx,
                                     full_mask, wpair_col)
println("M: $(size(M,1))×$(size(M,2))")

# 4. Rank → dimension of the eigenvector variety  (dim = rank - 1)
@time "QR(M)" r = sparse_qr_rank(M)
println("rank(M) = $r / $d")
println("dim of eigenvector variety = $(r - 1)/$n")

build W: 0.418883 seconds (438.35 k allocations: 2.823 GiB, 25.25% gc time, 171.90% compilation time)
W: 11440×11440
λ = -31.75542516293321 - 8.990422111400177im   (converged in 36 Arnoldi iters)
build M: 0.354473 seconds (440.04 k allocations: 3.029 GiB, 26.80% gc time, 182.87% compilation time: 5% of which was recompilation)
M: 11440×7260
QR(M): 10.260481 seconds (11.74 k allocations: 7.097 GiB, 0.40% gc time, 0.11% compilation time)
rank(M) = 7259 / 7260
dim of eigenvector variety = 7258/11440
